# 🏗️ YOLOv8 Construction Safety Detection — Training & Evaluation

This notebook provides a **complete, reproducible pipeline** for training a YOLOv8 model to detect safety-related objects on construction sites.

**Classes:** worker, helmet, no-helmet, vest, no-vest, machinery, vehicle, safety-barrier

---
### How to use:
1. Open in Google Colab (click the badge above or use File → Open in Colab)
2. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
3. **Runtime → Restart runtime → Run all**
4. Wait ~30–40 minutes for training to complete

> **No Roboflow account or API key required.** The dataset is stored directly in the repository.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install ultralytics==8.2.0 -q

# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import libraries
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

from IPython.display import Image, display
import os
import glob

## 2. Clone Repository & Access Dataset

In [ ]:
# ============================================================
# Clone the repository — dataset is inside 'images dataset/'
# IMPORTANT: Replace with YOUR GitHub repo URL
# ============================================================
REPO_URL = "https://github.com/musleh557790473-cell/Construction-Site-Safety-Monitoring-YOLOv8-Object-Detection.git"
REPO_NAME = "Construction-Site-Safety-Monitoring-YOLOv8-Object-Detection"  # folder name after cloning

# Clone if not already cloned
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"✅ Repository cloned successfully")
else:
    print(f"✅ Repository already exists, pulling latest...")
    !cd {REPO_NAME} && git pull

# Set dataset path
DATASET_DIR = os.path.join(REPO_NAME, "images dataset")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Exists: {os.path.exists(DATASET_DIR)}")

In [ ]:
# Verify dataset structure
DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

# Print data.yaml contents
with open(DATA_YAML, 'r') as f:
    print(f.read())

# Count images
train_imgs = len(glob.glob(os.path.join(DATASET_DIR, "train/images/*")))
val_imgs = len(glob.glob(os.path.join(DATASET_DIR, "valid/images/*")))
print(f"\nTrain images: {train_imgs}")
print(f"Validation images: {val_imgs}")
if train_imgs + val_imgs > 0:
    print(f"Split: {train_imgs/(train_imgs+val_imgs)*100:.0f}/{val_imgs/(train_imgs+val_imgs)*100:.0f}")

## 3. Fix data.yaml Paths for Colab

The `data.yaml` may contain relative paths that need updating for the Colab environment.

In [ ]:
import yaml

# Read current data.yaml
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

# Update paths to absolute Colab paths
abs_dataset_dir = os.path.abspath(DATASET_DIR)
data_config['train'] = os.path.join(abs_dataset_dir, 'train', 'images')
data_config['val'] = os.path.join(abs_dataset_dir, 'valid', 'images')

# Save updated data.yaml (Colab version)
COLAB_DATA_YAML = "data_colab.yaml"
with open(COLAB_DATA_YAML, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ Colab data.yaml created:")
with open(COLAB_DATA_YAML, 'r') as f:
    print(f.read())

## 4. Model Training

In [ ]:
# ============================================================
# Training Configuration
# ============================================================
MODEL_VARIANT = "yolov8n.pt"  # Nano variant (fastest)
EPOCHS = 50                    # Minimum 30 required
BATCH_SIZE = 16
IMG_SIZE = 640
PROJECT_NAME = "construction-safety"
RUN_NAME = "yolov8n-50ep"

# GPU check — fallback to verification run if no GPU
USE_GPU = torch.cuda.is_available()
if not USE_GPU:
    print("⚠️ No GPU detected! Running 5-epoch verification run...")
    print("   Pre-trained weights will be loaded for full inference.")
    EPOCHS = 5

In [ ]:
# Initialize model and train
model = YOLO(MODEL_VARIANT)

results = model.train(
    data=COLAB_DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True
)

print("\n✅ Training complete!")

## 5. Training Curves & Metrics

In [ ]:
# Display training curves
RESULTS_DIR = f"{PROJECT_NAME}/{RUN_NAME}"

curves = [
    "results.png",
    "confusion_matrix.png",
    "F1_curve.png",
    "PR_curve.png",
    "P_curve.png",
    "R_curve.png"
]

for curve in curves:
    path = f"{RESULTS_DIR}/{curve}"
    if os.path.exists(path):
        print(f"\n📊 {curve}")
        display(Image(filename=path, width=800))
    else:
        print(f"⚠️ {curve} not found")

In [ ]:
# Print final metrics
import csv

results_csv = f"{RESULTS_DIR}/results.csv"
if os.path.exists(results_csv):
    with open(results_csv, 'r') as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        last = rows[-1]
        
        print("="*50)
        print("FINAL TRAINING METRICS")
        print("="*50)
        for key, value in last.items():
            key = key.strip()
            if any(m in key for m in ['precision', 'recall', 'mAP']):
                print(f"  {key}: {float(value):.4f}")
        print("="*50)

## 6. Validation Evaluation

In [ ]:
# Load best weights and evaluate
BEST_WEIGHTS = f"{RESULTS_DIR}/weights/best.pt"

# ============================================================
# If using pre-trained weights (no GPU scenario), uncomment:
# BEST_WEIGHTS = "/content/best.pt"  # Upload your weights
# ============================================================

best_model = YOLO(BEST_WEIGHTS)

# Run validation
val_results = best_model.val(
    data=COLAB_DATA_YAML,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_NAME,
    name=f"{RUN_NAME}-val",
    exist_ok=True
)

print(f"\n📊 Validation Results:")
print(f"   Precision:    {val_results.box.mp:.4f}")
print(f"   Recall:       {val_results.box.mr:.4f}")
print(f"   mAP@50:       {val_results.box.map50:.4f}")
print(f"   mAP@50-95:    {val_results.box.map:.4f}")

## 7. Inference on Validation Images

In [ ]:
# Run predictions on validation set (sample of 10)
val_images_dir = os.path.join(DATASET_DIR, "valid", "images")
val_images = sorted(glob.glob(os.path.join(val_images_dir, "*")))[:10]

predictions = best_model.predict(
    source=val_images,
    imgsz=IMG_SIZE,
    conf=0.25,
    save=True,
    project=PROJECT_NAME,
    name=f"{RUN_NAME}-val-preds",
    exist_ok=True
)

# Display predictions
pred_dir = f"{PROJECT_NAME}/{RUN_NAME}-val-preds"
pred_images = sorted(glob.glob(f"{pred_dir}/*.jpg") + glob.glob(f"{pred_dir}/*.png"))

for img_path in pred_images[:10]:
    print(f"\n🖼️ {os.path.basename(img_path)}")
    display(Image(filename=img_path, width=600))

## 8. Inference on New Images

In [ ]:
# ============================================================
# Upload 5 new construction site images to Colab, or
# provide a folder path with new test images
# ============================================================

# Option 1: Upload files manually
# from google.colab import files
# uploaded = files.upload()

# Option 2: Use a test images folder
NEW_IMAGES_DIR = "test_images/"  # Create this folder and add images

if os.path.exists(NEW_IMAGES_DIR) and os.listdir(NEW_IMAGES_DIR):
    new_predictions = best_model.predict(
        source=NEW_IMAGES_DIR,
        imgsz=IMG_SIZE,
        conf=0.25,
        save=True,
        project=PROJECT_NAME,
        name=f"{RUN_NAME}-new-preds",
        exist_ok=True
    )
    
    new_pred_dir = f"{PROJECT_NAME}/{RUN_NAME}-new-preds"
    new_pred_images = sorted(glob.glob(f"{new_pred_dir}/*.jpg") + glob.glob(f"{new_pred_dir}/*.png"))
    
    for img_path in new_pred_images[:5]:
        print(f"\n🖼️ {os.path.basename(img_path)}")
        display(Image(filename=img_path, width=600))
else:
    print("⚠️ No new images found. Please upload images to 'test_images/' folder.")
    print("   You can use the Colab file upload or mount Google Drive.")

## 9. Export Weights

In [ ]:
# Save best weights for download
import shutil

if os.path.exists(BEST_WEIGHTS):
    shutil.copy(BEST_WEIGHTS, "best.pt")
    print(f"✅ Weights saved to best.pt ({os.path.getsize('best.pt')/1e6:.1f} MB)")
    print("   Download and upload to GitHub Releases.")
    
    # Optionally download to local machine
    # from google.colab import files
    # files.download('best.pt')

## 10. Summary

| Item | Value |
|------|-------|
| Model | YOLOv8n |
| Epochs | 50 |
| Batch Size | 16 |
| Image Size | 640 |
| Precision | _update_ |
| Recall | _update_ |
| mAP@50 | _update_ |
| mAP@50-95 | _update_ |

**Next steps:** Upload weights to GitHub Releases and update the README with final metrics.